# 1. Imports

In [ ]:
import sys
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.axes import Axes
from scipy.stats import gaussian_kde
import altair as alt
alt.data_transformers.enable("vegafusion")
# alt.data_transformers.enable('json')

sys.path.append("../")
from src.plot import donut_chart
from src.subset_visualization import plot_given_genes_on_feature_space

plt.style.use("../../config/DIT_HAP.mplstyle")
COLORS = plt.rcParams['axes.prop_cycle'].by_key()['color']
AX_WIDTH, AX_HEIGHT = plt.rcParams['figure.figsize']

# 2. Load data

In [ ]:
gene_names = pd.read_csv("../../resources/pombase_data/2025-10-01/Gene_metadata/gene_IDs_names_products.tsv", sep="\t")
gene_names["gene_name"] = gene_names["gene_name"].fillna(gene_names["gene_systematic_id"])
sysID2name = gene_names.set_index("gene_systematic_id")["gene_name"].to_dict()

DIT_HAP_data = pd.read_csv("../../results/HD_DIT_HAP_generationRAW/18_gene_level_clustering/kmeans_cluster_result.tsv", sep="\t")
gRNA_data = pd.read_csv("../../resources/YES_HD_all_gRNAs_with_fitted_params_and_DIT-HAP.csv", low_memory=False).query("order_by_M_sum == 1").copy()

pombe_gene_features = pd.read_csv("../../resources/pombe_features/pombe_coding_gene_protein_features.tsv", sep="\t").rename(
    columns={
        "Gene dispensability. This study": "Deletionlibrary_essentiality"
    }
)

all_brite_pathways = pd.read_csv("../../resources/KEGG/combined_brite_table.tsv", sep="\t")

output_dir = Path("../../results/HD_DIT_HAP_generationRAW/23_pathway_analysis")
output_dir.mkdir(parents=True, exist_ok=True)

# 3. Merge all phenotype features

In [ ]:
merged_fitness_data = pombe_gene_features[["gene_systematic_id", 'FYPOviability', 'Deletionlibrary_essentiality', 'Category',
     'Barseq_from_dulab', 'Barseq_from_koch',
     'Integration density, in-vivo (integrations/kb/million inserts)',
     'ipkm', 'uipkm', 'colony_size_Malecki2016', 'Max Growth Rate',
     'Colony Formation']].merge(
          DIT_HAP_data[["Systematic ID", "Name", "um", "lam", "revised_cluster"]], 
          left_on="gene_systematic_id",
          right_on="Systematic ID",
          how="left"
     ).merge(
          gRNA_data[["Systematic ID", "um", "lam"]],
          left_on="gene_systematic_id",
          right_on="Systematic ID",
          suffixes=("_DIT_HAP", "_gRNA"),
          how="left"
     )
merged_fitness_data["Name"] = merged_fitness_data["gene_systematic_id"].map(sysID2name)

fitness_data = merged_fitness_data[
    [
        "gene_systematic_id", "Name",
        "Barseq_from_dulab", "Barseq_from_koch",
        "Integration density, in-vivo (integrations/kb/million inserts)",
        "ipkm", "uipkm",
        "colony_size_Malecki2016", "Max Growth Rate", "Colony Formation",
        "um_DIT_HAP", "um_gRNA"
    ]
].copy()
fitness_data["Integration density, in-vivo (integrations/kb/million inserts)"] = fitness_data["Integration density, in-vivo (integrations/kb/million inserts)"].clip(upper=200)
fitness_data["ipkm"] = fitness_data["ipkm"].clip(upper=200)
fitness_data["uipkm"] = fitness_data["uipkm"].clip(upper=200)

# 4. Scatter plot for pair-wise fitness comparison

In [ ]:
alt.Chart(fitness_data).mark_point(opacity=0.3, size=10).encode(
    x = alt.X(alt.repeat("row"), type="quantitative", scale=alt.Scale(zero=False)),
    y = alt.Y(alt.repeat("column"), type="quantitative", scale=alt.Scale(zero=False)),
    tooltip = ["Name", "gene_systematic_id"]
).repeat(
    row=[
        "Barseq_from_dulab", "Barseq_from_koch",
        "Integration density, in-vivo (integrations/kb/million inserts)",
        "ipkm", "uipkm",
        "colony_size_Malecki2016", "Max Growth Rate", "Colony Formation",
        "um_DIT_HAP", "um_gRNA",
    ],
    column=[
        "Barseq_from_dulab", "Barseq_from_koch",
        "Integration density, in-vivo (integrations/kb/million inserts)",
        "ipkm", "uipkm",
        "colony_size_Malecki2016", "Max Growth Rate", "Colony Formation",
        "um_DIT_HAP", "um_gRNA",
    ]
)

# Pathway info

In [ ]:
fitness_data_with_pathways = fitness_data.merge(
    all_brite_pathways.query("Level_2 != 'Prokaryotic type' "),
    on="Name",
    how="left"
).sort_values(
    [
        "Category",
        "Description",
        "Level_1",
        "Level_2",
        "Level_3",
        "Level_4",
        "Level_5",
        "Level_6",
        "Level_7"
    ]
)

In [ ]:
print("Genes: ", fitness_data_with_pathways.query("um_DIT_HAP.notna() and Level_1.notna()")["Name"].nunique())
print("level_3: ", fitness_data_with_pathways["Level_3"].nunique())
print("level_4: ", fitness_data_with_pathways["Level_4"].nunique())

In [ ]:
df = fitness_data_with_pathways.query("um_DIT_HAP.notna() and Level_1.notna()")

level_category = "Level_3"
select_group = alt.selection_point(fields=[level_category], bind='legend')
varible_paris = [
    ("um_DIT_HAP", "um_gRNA"),
    ("um_gRNA", "um_DIT_HAP"),
]

for desc, sub_df in df.groupby("Description"):
    print(f"Generating plot for pathway: {desc}...")

    sub_df = sub_df.drop_duplicates(subset=["Name", "Level_3", "Level_4"])

    charts = []
    for level, level_df in sub_df.groupby(level_category):
        print(f"  Description: {level}...")
        base = alt.Chart(level_df).mark_circle(size=20, opacity=0.7)

        gaussian_jitter = [
            base.transform_calculate(
                bin_index=f"floor((datum['{x_col}']) / 0.05)"
            ).transform_joinaggregate(
                um_bin_count="count()",
                groupby=["bin_index", "Level_4"]
            ).transform_calculate(
                adj_um_bin_count = "max(0, datum.um_bin_count - 1)"
            ).transform_calculate(
                jitter="sqrt(-2*log(random()))*cos(2*PI*random())* min(datum.adj_um_bin_count, 10) / 10",
                # jitter="((random() + random() + random() + random()) - 2) / 2 * min(datum.adj_um_bin_count, 10) / 10"
            ).encode(
                x=alt.X(x_col, type="quantitative", scale=alt.Scale(domain=(-0.2, 1.6), clamp=True)),
                y=alt.Y("Level_4:N", axis=alt.Axis(title="Pathway (level 4)", labelLimit=300, grid=True, tickBand="extent")),
                yOffset=alt.YOffset("jitter:Q"),
                color=alt.Color(c_col, type="quantitative", scale=alt.Scale(scheme="reds", domainMin=-0.3, domainMid=0, domainMax=1.4), legend=alt.Legend(title="um")),
                opacity=alt.condition(select_group, alt.value(1), alt.value(0)),
                tooltip=["Name", "um_DIT_HAP", "um_gRNA", "Max Growth Rate", "Level_7", "Level_8"],
            ).resolve_scale(
                y='shared'
            ).add_params(
                select_group
            ).transform_filter(
                select_group
            ).properties(
                title=level + f" (n={level_df['Name'].nunique()})",
                height=alt.Step(30)
            ) for x_col, c_col in varible_paris
        ]
        charts.append(alt.hconcat(*gaussian_jitter))
    alt.vconcat(*charts).save(output_dir / f"{desc.replace(' ', '_')}_pathway.html")

In [ ]:
enriched_terms = enrichment_res.query("Cluster != 9 and namespace != 'MF' and pop_count < 400")["term_id"].unique()

In [ ]:
from matplotlib.lines import Line2D

cluster_colors = {
    1: '#d57fbd',
    2: '#e0788f',
    3: '#dd8369',
    4: '#c4954b',
    5: '#98a64e',
    6: '#4aadce',
    7: '#6b99df',
    8: '#a78bd9',
    9: '#64af6d',
}

def create_gradient_colormap(color: str, name: str) -> LinearSegmentedColormap:
    """Create a gradient colormap from a single color."""
    colors = ['white', color]
    n_bins = 256
    cmap = LinearSegmentedColormap.from_list(name, colors, N=n_bins)
    return cmap

def plot_term_genes(
    ax: Axes,
    data_df: pd.DataFrame,
    cluster_colors: dict[int, str],
    genes: list[str],
    gene_column: str,
    x_feature: str = "um",
    y_feature: str = "lam",
    **kwargs
) -> Axes:
    """ Plot given genes on feature space. """

    # all points in light gray
    data_df = data_df.dropna(subset=[x_feature, y_feature], how='any')
    x_all = data_df[x_feature]
    y_all = data_df[y_feature]
    ax.scatter(x_all, y_all, color='lightgray', alpha=0.4, **kwargs, zorder=0)

    legend_handles = []
    for cluster, cluster_genes in data_df.query(f"{gene_column} in @genes").groupby("revised_cluster"):
        cluster = int(cluster)
        color = cluster_colors.get(cluster, 'gray')
        cmap = create_gradient_colormap(color, name=f'cluster_{cluster}_cmap')

        # points for given genes
        subset_df = cluster_genes.dropna(subset=[x_feature, y_feature])
        x_subset = subset_df[x_feature]
        y_subset = subset_df[y_feature]

        try:
            xy_subset = np.vstack([x_subset, y_subset])
            z = gaussian_kde(xy_subset)(xy_subset)
            ax.scatter(x_subset, y_subset, c=z, cmap=cmap, **kwargs, zorder=1)
        except Exception:
            ax.scatter(x_subset, y_subset, color=color, **kwargs, zorder=1)
        
        # Create legend handle for this cluster
        handle = Line2D([0], [0], marker='o', color='w', 
                       markerfacecolor=color, markersize=10, alpha=1.0,
                       label=f"Cluster {cluster} (n={len(subset_df)})")
        legend_handles.append(handle)
    
    ax.set_xlabel(x_feature)
    ax.set_ylabel(y_feature)
    ax.grid(True)
    
    # Add legend to this specific axis
    ax.legend(handles=legend_handles, fontsize=8)
    ax.set_xlim(-0.2, 1.8)
    ax.set_ylim(-0.3, 14)

    return ax


In [ ]:
with PdfPages(output_dir / "GO_enrichment_term_all_genes_analysis_plots.pdf") as pdf:
    for term in enriched_terms:
        term_df = enrichment_res.query("term_id == @term")
        term_name = term_df["term"].values[0]
        ns = term_df["namespace"].values[0]
        print(f"Generating plot for GO term: {term} - {term_name}...")
        term_genes = term_df["pop_items"].values[0].split(",")
        enriched_clusters = term_df["Cluster"].unique().tolist()
        fig, axes = plt.subplots(1,2,figsize=(2*AX_WIDTH, AX_HEIGHT), sharex=True, sharey=True)

        plot_term_genes(
            ax=axes[0],
            data_df=merged_fitness_data,
            genes = term_genes,
            gene_column = "Name",
            x_feature="um_DIT_HAP",
            y_feature="lam_DIT_HAP",
            cluster_colors=cluster_colors,
            s=20,
        )
        plot_term_genes(
            ax=axes[1],
            data_df=merged_fitness_data,
            genes = term_genes,
            gene_column = "Name",
            x_feature="um_gRNA",
            y_feature="lam_gRNA",
            cluster_colors=cluster_colors,
            s=20,
        )

        fig.suptitle(f"{ns}: {term_name}\nEnriched in clusters: {','.join(map(str, enriched_clusters))}", y=1.02)

        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)